In [4]:
import os
import json

In [3]:
# (outdated_exploratory_trajectories, exploratory_trajectories, ground_truth_trajectories)
no_scenario_folder_pairs=[("/proj/m3benchmark/m3data/0905/outdated_balanced_rest_v4_exploratory_trajectory/trajectories","/proj/m3benchmark/m3data/0905/balanced_rest_v4_exploratory_trajectory/trajectories","/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2_after_generate/final"), # Multi-turn without scenarios exploratory
              ("/proj/m3benchmark/m3data/0905/outdated_m3_train_test_ood_rest_v2_single_turn_exploratory_trajectory/trajectories","/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2_single_turn_exploratory_trajectory/trajectories","/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2_single_turn_after_generate/final"), # Single Turn without scenarios exploratory
]
scenarios_folder_pairs=[("/proj/m3benchmark/m3data/0905/outdated_m3_train_test_ood_rest_v2_chunked_scenarios_exp/trajectories","/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2_chunked_scenarios_exp/trajectories","/proj/m3benchmark/danish/m3data/0905/m3_train_test_ood_rest_v2_chunked_scenarios_gt/final"), # Multi-turn scenarios exploratory
              ("/proj/m3benchmark/m3data/0905/outdated_m3_train_test_ood_rest_v2_single_turn_chunked_scenarios_st_exp/trajectories","/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2_single_turn_chunked_scenarios_st_exp/trajectories","/proj/m3benchmark/m3data/0905/m3_train_test_ood_rest_v2_single_turn_chunked_scenarios_st_gt/final"), # Single Turn scenarios exploratory
]

In [5]:
def get_scenario(data,sample_id,no_scenario=None):
    if no_scenario:
        return {"tool_use_policy": None, "policy_domain": None, "missing_api": None, "tool_availability": None}

    for item in data:
        if item["sample_id"] == sample_id:
            return item["scenarios"]

def check_folders(foldename):
    assert os.path.isdir(foldename)

def check_scenario_addition(out_folder, folder):
    assert len(os.listdir(out_folder)) == len(os.listdir(folder))
    filenames = os.listdir(folder)
    for filename in filenames:
        with open(f"{out_folder}/{filename}",'r') as f:
            out_data=json.load(f)
        with open(f"{folder}/{filename}",'r') as f:
            data=json.load(f)
        for key in data.keys():
            if key !="scenarios":
                assert out_data.get(key) == data.get(key)

In [34]:
# No Scenarios
for (out_folder, folder, _ ) in no_scenario_folder_pairs:
    filenames=os.listdir(out_folder)
    check_folders(out_folder)
    check_folders(folder)
    assert len(os.listdir(folder)) == 0
    for filename in filenames:
        with open(f"{out_folder}/{filename}",'r') as f:
            data=json.load(f)
        data["scenarios"]= get_scenario(data,None,True)
        with open(f"{folder}/{filename}", "w") as f:
            json.dump(data, f)

In [40]:
# With Scenarios
for (out_folder, folder, gt_folder) in scenarios_folder_pairs:
    filenames=os.listdir(out_folder)
    check_folders(out_folder)
    check_folders(folder)
    check_folders(gt_folder)
    assert len(os.listdir(folder)) == 0    
    for filename in filenames:
        with open(f"{out_folder}/{filename}",'r') as f:
            data=json.load(f)
        domain, sample_id = data["domain"], data["sample_id"]
        with open(f"{gt_folder}/{domain}_multiturn_bird_chunked_final.json","r") as f:
            gt_data=json.load(f)
        
        data["scenarios"]= get_scenario(gt_data,sample_id=sample_id,no_scenario=None)
        with open(f"{folder}/{filename}", "w") as f:
            json.dump(data, f)

In [ ]:
# Check correct
for (out_folder, folder, gt_folder) in scenarios_folder_pairs:
    check_scenario_addition(out_folder, folder)